In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import time

# 페이지 더보기
def load_all_cards(driver):
    while True:
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, 'button.more')
            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(1)
        except NoSuchElementException:
            print('마지막 페이지 도달')
            break



link = []

url = 'https://card-search.naver.com/list?companyCode=SS&brandNames=&sortMethod=ri&isRefetch=true&bizType=CPC'

driver = wb.Chrome()
driver.get(url)

load_all_cards(driver)

card_list = driver.find_element(By.CSS_SELECTOR, 'div.tabpanel > ul.list')
cards = card_list.find_elements(By.CSS_SELECTOR, 'li')

for card_link in cards:
    a_tag = card_link.find_element(By.CSS_SELECTOR, 'div.info > a.anchor')
    link.append(a_tag.get_attribute("href"))


rows = []
benefit_rows = []

for url in link:
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'lxml')

    # 카드명
    card_name_tags = soup.select_one('div.cardname')
    card_name = card_name_tags.get_text(strip=True) if card_name_tags else '카드명 없음'

    # 공지
    card_noti_tags = soup.select_one('dl.list b.noti')
    card_noti = card_noti_tags.get_text(strip=True) if card_noti_tags else '공지 없음'

    # 연회비
    card_dues_tags = soup.select_one('dd.desc.as_annualFee span.txt')
    card_dues = card_dues_tags.get_text(strip=True) if card_dues_tags else '연회비 없음'

    # 보상
    card_rewards_tags = soup.select_one('dd.desc.as_annualFee b')
    card_rewards = card_rewards_tags.get_text(strip=True) if card_rewards_tags else '보상 없음'

    # 기준 실적
    card_base_record_tags = soup.select_one('dd.desc.as_annualFee span')
    card_base_record = (
        card_base_record_tags.get_text(strip=True)
        if card_base_record_tags else '기준 실적 없음'
    )

    row = {
        '카드명': card_name,
        '연회비': card_dues,
        '보상': card_rewards,
        '기준실적': card_base_record,
        '공지': card_noti,
        'url': url,
   
    }

    rows.append(row)
  

    
    benefit_blocks = soup.select('div.Benefits details.details')

    for block in benefit_blocks:
        title_tag = block.select_one('h5.name b.text')
        desc_tag = block.select_one('h5.name i.desc')

        title = title_tag.get_text(strip=True) if title_tag else None
        desc = desc_tag.get_text(strip=True) if desc_tag else None
        
        
        benefit_rows.append({
        'card_name': card_name,
        '혜택카테고리': title,
        '혜택설명': desc
    })

  
df = pd.DataFrame(rows)
df_benfits = pd.DataFrame(benefit_rows)



마지막 페이지 도달


In [2]:
df_benfits

,card_name,혜택카테고리,혜택설명
0,삼성카드 taptap O,연회비지원,행사 응모필수!온라인신규회원 연회비 100% 캐시백(삼성개인신용카드 신규발급 시 제공)
1,삼성카드 taptap O,통신,이동통신요금 연 최대 6만원 할인이동통신요금 10% 할인
2,삼성카드 taptap O,대중교통,"교통 연 최대 6만원 할인버스, 지하철, 택시 10% 할인"
3,삼성카드 taptap O,영화,"영화 연 최대 6만원 할인CGV·롯데시네마 5,000원 할인"
4,삼성카드 taptap O,포인트/캐시백,쇼핑 7% 할인 및 1% 적립해외 1.3% 적립
...,...,...,...
394,CU·배달의민족 삼성카드 taptap,대중교통,"대중교통 1,000원당 100원 할인"
395,삼성카앤모아카드,주유,모든주유소 리터당 60원 할인주유포인트 0.2%·0.4% 적립
396,삼성카앤모아카드,오토,엔진오일 교환시 1만5천원 현장할인프라이버시콜(안심주차번호서비스) 무료
397,삼성카앤모아카드,영화,CGV 동반 1인 50% 현장할인


In [3]:
df

,카드명,연회비,보상,기준실적,공지,url
0,삼성카드 taptap O,"국내 1만원, 해외 1만원",신규 회원 최대 100% 지급,"국내 1만원, 해외 1만원",연 최대 36만원 할인 혜택!,https://card-search.naver.com/item?cardAdId=1530
1,삼성 iD SIMPLE 카드,"국내 7천원, 해외 7천원",신규 회원 최대 100% 지급,"국내 7천원, 해외 7천원",국내외 실적/한도없이 최대1% 할인,https://card-search.naver.com/item?cardAdId=10157
2,삼성 iD GLOBAL 카드,"국내 2만원, 해외 2만원",보상 없음,"국내 2만원, 해외 2만원",해외 이용 수수료 100% 할인,https://card-search.naver.com/item?cardAdId=10343
3,삼성 iD ON 카드,"국내 2만원, 해외 2만원, 가족 1만3천원",신규 회원 최대 100% 지급,"국내 2만원, 해외 2만원, 가족 1만3천원",연 최대 48만원 할인!,https://card-search.naver.com/item?cardAdId=10102
4,삼성 iD SELECT ALL 카드,"국내 2만원, 해외 2만원",신규 회원 최대 100% 지급,"국내 2만원, 해외 2만원",생활요금 할인 내게 맞춰 SELECT,https://card-search.naver.com/item?cardAdId=10522
5,모니모카드,"국내 1만원, 해외 1만원",보상 없음,"국내 1만원, 해외 1만원","나만의 혜택레시피, 연최대36만원할인",https://card-search.naver.com/item?cardAdId=10238
6,네이버페이 taptap,"국내 1만5천원, 해외 1만5천원",보상 없음,"국내 1만5천원, 해외 1만5천원",온라인도 오프라인도 N페이포인트 적립,https://card-search.naver.com/item?cardAdId=2776
7,삼성 iD ENERGY 카드,"국내 2만원, 해외 2만원",보상 없음,"국내 2만원, 해외 2만원",주유 건별 1만원 결제일할인 혜택,https://card-search.naver.com/item?cardAdId=10126
8,taptap DRIVE,"국내 1만원, 해외 1만원",신규 회원 최대 100% 지급,"국내 1만원, 해외 1만원",주유소 리터당 최대 150원 할인!,https://card-search.naver.com/item?cardAdId=3881
9,신세계이마트 삼성카드 7,"국내 1만8천원, 해외 2만원",보상 없음,"국내 1만8천원, 해외 2만원",신세계백화점 5% 할인쿠폰 제공,https://card-search.naver.com/item?cardAdId=277
